In [1]:
!pip install -q sentence-transformers

In [2]:
docs = [
    "You can buy a subscription directly from the pricing page using a credit card.",
    "Refunds are processed within 5-7 business days after a cancellation request.",
    "Our support team is available 24/7 via live chat and email.",
    "To reset your password, click 'Forgot Password' on the login screen.",
    "We offer a 14-day free trial with full access to all premium features.",
    "Enterprise plans include dedicated account managers and custom SLAs.",
    "You can cancel your plan anytime from the account settings page.",
    "Two-factor authentication can be enabled under Security Settings.",
    "Our API rate limit is 1000 requests per minute on the Pro plan.",
    "Invoices are automatically emailed at the start of each billing cycle.",
    "Team members can be invited via email from the Workspace settings.",
    "Data is encrypted at rest using AES-256 encryption.",
    "We support integrations with Slack, Zapier, and Google Workspace.",
    "Mobile apps are available for both iOS and Android devices.",
    "Storage limits vary by plan, ranging from 5GB to unlimited.",
    "You can export your data anytime as a CSV or JSON file.",
    "Our uptime SLA guarantees 99.9% availability for Enterprise customers.",
    "Custom domains can be configured under Domain Settings.",
    "We do not store your credit card number on our servers.",
    "Notifications can be customized in the Preferences menu.",
]
print(f"{len(docs)} docs loaded.")

20 docs loaded.


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

doc_store = []  # list of (text, vector) tuples
for doc in docs:
    vec = model.encode(doc)
    doc_store.append((doc, vec))

print(f"Embedded {len(doc_store)} documents. Vector dim: {doc_store[0][1].shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedded 20 documents. Vector dim: (384,)


In [4]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_search(query, top_k=3):
    query_vec = model.encode(query)
    scored = [(text, cosine_similarity(query_vec, vec)) for text, vec in doc_store]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

In [5]:
def keyword_search(query, top_k=3):
    query_lower = query.lower()
    matches = [text for text, _ in doc_store if query_lower in text.lower()]
    return matches[:top_k]

In [6]:
query = "How do I purchase a plan?"  # note: docs use "buy", not "purchase"

print("=== Semantic Search ===")
for text, score in semantic_search(query):
    print(f"{score:.3f}  {text}")

print("\n=== Keyword Search ===")
results = keyword_search(query)
print(results if results else "No matches found.")

=== Semantic Search ===
0.507  You can buy a subscription directly from the pricing page using a credit card.
0.480  You can cancel your plan anytime from the account settings page.
0.381  Enterprise plans include dedicated account managers and custom SLAs.

=== Keyword Search ===
No matches found.
